In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report,r2_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
import xgboost as xgb

In [2]:
# Load the PCA loadings file
loadings = pd.read_csv('pca_loadings_with_features.csv', index_col=0)

In [3]:
print("Loaded Loadings Shape:", loadings.shape)

# Select top features from first 5 PCs
top_features = set()
for pc in loadings.columns[:5]:   # PC1 to PC5
    top = loadings[pc].abs().nlargest(8).index.tolist()
    top_features.update(top)

Loaded Loadings Shape: (21, 17)


In [4]:
top_features = list(top_features)
print("\nSelected Important Features from PCA Loadings:")
print(top_features)


Selected Important Features from PCA Loadings:
['cash_flow_3m_trend', 'Loan_Term_Months', 'cash_flow_6m_trend', 'Credit_Score', 'Debt_Service_Coverage', 'monthly_sales', 'LTV_Ratio', 'gst_delay_days', 'Current_Ratio', 'Annual_Revenue', 'days_past_due', 'EBITDA', 'Interest_Rate', 'industry_risk_score', 'Loan_Amount', 'annual_turnover', 'Collateral_Value', 'Delinquency_Days']


In [23]:
# Now use these features for modeling
df_data = pd.read_csv('preprocessed_scaled_commercial_loans.csv')

X = df_data[top_features]
y = df_data['Default']

print("Features used for training:", X.shape[1])

Features used for training: 18


In [24]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [25]:
# Train XGBoost
xgboost = xgb.XGBClassifier(n_estimators=300, learning_rate=0.1, max_depth=6, random_state=42)
xgboost.fit(X_train, y_train)

y_pred = xgboost.predict(X_test)

In [26]:
print("Classification Report:")
print(classification_report(y_test, y_pred))
r2scores = r2_score(y_test, y_pred)
print("R2Score for XGBoost:",r2scores)

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1710
           1       1.00      1.00      1.00       290

    accuracy                           1.00      2000
   macro avg       1.00      1.00      1.00      2000
weighted avg       1.00      1.00      1.00      2000

R2Score for XGBoost: 1.0


In [27]:
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)
y_pred = dt.predict(X_test)

print("Classification Report:")
print(classification_report(y_test, y_pred))
r2scores = r2_score(y_test, y_pred)
print("R2Score for DecisionTree:",r2scores)

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1710
           1       1.00      1.00      1.00       290

    accuracy                           1.00      2000
   macro avg       1.00      1.00      1.00      2000
weighted avg       1.00      1.00      1.00      2000

R2Score for DecisionTree: 1.0


In [28]:
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)
y_pred = lr.predict(X_test)

print("Classification Report:")
print(classification_report(y_test, y_pred))
r2scores = r2_score(y_test, y_pred)
print("R2Score for LogisticRegression:",r2scores)

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1710
           1       1.00      1.00      1.00       290

    accuracy                           1.00      2000
   macro avg       1.00      1.00      1.00      2000
weighted avg       1.00      1.00      1.00      2000

R2Score for LogisticRegression: 1.0


C:\Anaconda\envs\AI\lib\site-packages\sklearn\linear_model\logistic.py:432: FutureWarning: Default solver will be changed to 'lbfgs' in 0.22. Specify a solver to silence this warning.
  FutureWarning)


In [29]:
rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

print("Classification Report:")
print(classification_report(y_test, y_pred))
r2scores = r2_score(y_test, y_pred)
print("R2Score for RandomForest:",r2scores)

C:\Anaconda\envs\AI\lib\site-packages\sklearn\utils\fixes.py:230: DeprecationWarning: distutils Version classes are deprecated. Use packaging.version instead.
  if _joblib.__version__ >= LooseVersion('0.12'):


Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1710
           1       1.00      1.00      1.00       290

    accuracy                           1.00      2000
   macro avg       1.00      1.00      1.00      2000
weighted avg       1.00      1.00      1.00      2000

R2Score for RandomForest: 1.0


C:\Anaconda\envs\AI\lib\site-packages\sklearn\utils\fixes.py:230: DeprecationWarning: distutils Version classes are deprecated. Use packaging.version instead.
  if _joblib.__version__ >= LooseVersion('0.12'):


In [30]:
# XGBoost, DecisionTree and RandomForest are Equally good Model for this DataSet

import pickle
filename = 'final_commercial_loans_ews_model.pkl'
with open(filename, 'wb') as file:
    pickle.dump(xgboost, file)